#Load Environment

In [ ]:
import csv
import requests
#using beautifulesoup to scrape the site
try:
    from bs4 import BeautifulSoup
except ImportError:
    !pip install beautifulsoup4
    from bs4 import BeautifulSoup
from bs4 import BeautifulSoup
import re
# Install the necessary libraries
!pip install google-auth google-auth-oauthlib google-auth-httplib2 google-api-python-client
# Import the libraries
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

#Connect to Google Drive

In [ ]:
# Optional Colab-only setup: mount your own Google Drive if needed.
# from google.colab import drive
# drive.mount("/content/drive")


#Create path for saving scraped files

In [ ]:
from pathlib import Path
import os

# Portable output workspace; override SJK_PORTFOLIO_DIR to use a different folder.
PROJECT_DIR = Path(os.environ.get('SJK_PORTFOLIO_DIR', '/content/sjk_portfolio')) / 'Yeti_Help_File_Scrape'
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
path = str(PROJECT_DIR)


#Save URL's to CSV for scraping

In [ ]:
import csv
import requests
from bs4 import BeautifulSoup

def pagesource(url):
  # Get the page source
  response = requests.get(url)
  soup = BeautifulSoup(response.text, "html.parser")
  return soup

# Set the base URL and keyword
base_url = "https://intercom.help"
keyword = "collections"

# Set the URL to scrape
url = base_url + "/yetiacademy/en/"

# Get the page source
soup = pagesource(url)

# Find all the links on the page
links = soup.find_all('a')

# Flag to track whether we are within the desired range of URLs
in_range = False

# Open the CSV file in write mode
with open(str(PROJECT_DIR / 'HelpContentStep1.csv'), 'w', newline='') as csvfile:
  # Create a CSV writer object
  writer = csv.writer(csvfile)

  # Iterate through the links
  for link in links:
    try:
      # Extract the href attribute of the link
      href = link['href']
      print(href)

      # Add the base URL to the URL
      full_url = base_url + href

      # Check if the URL is the first /en/ URL
      if full_url == url:
        # Set the in_range flag to True
        in_range = True
      # Check if the in_range flag is set to True and the URL is not the first /en/ URL
      elif in_range and full_url != url:
        # Check if the URL contains the keyword
        if keyword in full_url:
          # Write the URL to the CSV file
          writer.writerow([full_url])

          # Print the full URL
          print(full_url)
    except:
      # Ignore any errors that might occur
      pass

#Find and save all help file URL's from step 1

In [ ]:
import csv
import requests
from bs4 import BeautifulSoup

def pagesource(url):
  # Get the page source
  response = requests.get(url)
  soup = BeautifulSoup(response.text, "html.parser")
  return soup

# Set the keyword to be included in the URLs
keyword = "articles"

# Open the CSV file in read mode
with open(str(PROJECT_DIR / 'HelpContentStep1.csv'), 'r') as csvfile:
  # Create a CSV reader object
  reader = csv.reader(csvfile)

  # Initialize the list of URLs
  urls = []

  # Iterate through the rows in the CSV file
  for row in reader:
    # Get the URL from the row
    url = row[0]

    # Add the URL to the list
    urls.append(url)

# Format the list of URLs as a string
urls_str = "\n".join(f"    \"{url}\"," for url in urls)

# Open the CSV file in write mode
with open(str(PROJECT_DIR / 'HelpContentStep2.csv'), 'w', newline='') as csvfile:
  # Create a CSV writer object
  writer = csv.writer(csvfile)

  # Iterate through the URLs
  for url in urls:
    # Send a GET request to the URL
    response = requests.get(url)

    # Parse the HTML content of the page
    soup = BeautifulSoup(response.content, "html.parser")

    # Find all the links on the page
    links = soup.find_all('a')

    # Iterate through the links
    for link in links:
      try:
        # Extract the href attribute of the link
        href = link['href']

        # Print the full URL
        print(full_url) #<--uncomment for testing

        # Add the base URL to the URL
        full_url = "https://intercom.help" + href

        # Write the URL to the CSV file if it contains the keyword
        if re.search(keyword, full_url):
          writer.writerow([full_url])


      except:
        # Ignore any errors that might occur
        pass

#Save article content under each H2/H3 header to article column

In [ ]:
import requests
from bs4 import BeautifulSoup
import csv
import os

# Build the full path to the CSV file using the path variable
csv_path = os.path.join(path, 'YetiHelpContentFinal.csv')

# Open the CSV file in write mode
with open(csv_path, 'w', newline='', encoding='utf-8') as csvfile:
    # Create a CSV writer object
    writer = csv.writer(csvfile)
    # Write the header row to the CSV file
    writer.writerow(['title', 'href', 'description', 'subject', 'h2', 'article'])

    # Open the CSV file in read mode
    with open(os.path.join(path, 'HelpContentStep2.csv'), 'r') as csvfile:
        # Create a CSV reader object
        reader = csv.reader(csvfile)
        # Iterate through the rows of the CSV file
        for row in reader:
            # The URL is in the first column of each row
            url = row[0]
            # Send a GET request to the URL and get the response
            response = requests.get(url)
            # Parse the HTML content of the page
            soup = BeautifulSoup(response.text, "html.parser")

            # Extract the title of the page
            title_element = soup.find('meta', {'property': 'og:title'})
            title = title_element['content'] if title_element else ''

            # Locate all the elements with the following xpath
            href_elements = soup.find_all('link', {'rel': 'alternate'})
            href = href_elements[0]['href'] if href_elements else ''

            # Try to locate the description element
            description_element = soup.find('meta', {'property': 'og:description'})
            description = description_element['content'] if description_element else ''

            # Try to locate the article element
            article_element = soup.find('article')
            article = article_element.text if article_element else ''

            # Find all the H2 headers in the article element and extract the subject
            h2_elements = article_element.find_all(['h2', 'h3'])
            subject_element = soup.find('h1', {'class': 'page-title'})
            subject = subject_element.get_text() if subject_element else ''

            # If we don't find a subject, try to get it from the title
            if not subject:
                subject = title

            # Iterate through the H2 headers
            for h2 in h2_elements:
                # Extract the text of the H2 header
                h2_text = h2.get_text()
                # Find the article content for the current H2 header
                article_content = ""
                next_sibling = h2.next_sibling
                while next_sibling and next_sibling.name not in ['h2', 'h3']:
                    article_content += next_sibling.get_text()
                    next_sibling = next_sibling.next_sibling
                # Write the row to the CSV file
                writer.writerow([title, href, description, subject, h2_text, article_content])
                # Print the row to the console
                print(title, href, description, subject, h2_text, article_content)